# 05 Adavanced Models

Gradient Boosting (e.g., XGBoost or LightGBM). 
Perform light hyperparameter tuning (depth, learning rate, n_estimators). 

## Setup

In [9]:
# %pip install lightgbm

import os, glob
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score

from lightgbm import LGBMRegressor
# from xgboost import XGBRegressor

pd.set_option('display.max_columns', 100)

DATA = sorted(glob.glob('../../data/cleaned_CRMLSSold*.csv'))[-1]
RESULTS = '../../results/model_results.csv'

df = pd.read_csv(DATA, parse_dates=[DATE])
print(f"loaded {DATA}  ->  {df.shape[0]:,} rows x {df.shape[1]} columns")

TARGET = 'ClosePrice'
DATE = 'CloseDate'

TEST_MONTH = (pd.read_csv(DATA, usecols=[DATE], parse_dates=[DATE])[DATE].dt.to_period("M").astype(str).max())
MAX_TRAIN_MONTHS = 11 

avail_data = (pd.read_csv(DATA, usecols=[DATE], parse_dates=[DATE])[DATE].dt.to_period("M").nunique() - 1)
TRAIN_MONTHS = min(avail_data, MAX_TRAIN_MONTHS)

RANDOM_STATE = 42 #needed for decision tree and random forest
print(f"Using {DATA}  |  test month = {TEST_MONTH}  |  train window = {TRAIN_MONTHS} months")

WEEK = 7

C:\Users\Vivian\AppData\Local\Temp\ipykernel_2400\1146827666.py:26: DtypeWarning: Columns (0: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA, parse_dates=[DATE])


loaded ../../data\cleaned_CRMLSSold202506_202606.csv  ->  142,750 rows x 1567 columns
Using ../../data\cleaned_CRMLSSold202506_202606.csv  |  test month = 2026-06  |  train window = 11 months


In [10]:
#split data into train/test sets

def chrono_split(data, test_month=TEST_MONTH, window=TRAIN_MONTHS, date=DATE):

      months = data[date].dt.to_period("M")
      test_p = pd.Period(test_month, freq="M")
      train_start = test_p - window 

      test = data[months == test_p]
      train = data[(months >= train_start) & (months < test_p)]
      # Error checks
      assert len(train) > 0 and len(test) > 0, "ERROR: Empty split (check params)"
      assert train[date].max() < test[date].min(), "Chronology violated"
      return train, test

train_df, test_df = chrono_split(df)

print(f"Train: {train_df.shape[0]:,} rows "f"({train_df[DATE].min().date()} --> {train_df[DATE].max().date()})")
print(f"Test:  {test_df.shape[0]:,} rows "f"({test_df[DATE].min().date()} --> {test_df[DATE].max().date()})")

Train: 118,247 rows (2025-07-01 --> 2026-05-31)
Test:  12,837 rows (2026-06-01 --> 2026-06-30)


In [11]:
# trim to train 1/99 bounds
def trim_to_train_bounds(train, test, target=TARGET, low=0.01, high=0.99):
    lo, hi = train[target].quantile(low), train[target].quantile(high)
    return (train[train[target].between(lo, hi)].copy(),
            test[test[target].between(lo, hi)].copy(), lo, hi)

test_df_full = test_df.copy()
_ntr, _nte = len(train_df), len(test_df)
train_df, test_df, TRIM_LO, TRIM_HI = trim_to_train_bounds(train_df, test_df)
print(f"Trim bounds (train 1/99): ${TRIM_LO:,.0f} - ${TRIM_HI:,.0f}")
print(f"Train: {_ntr:,} -> {len(train_df):,} | Test: {_nte:,} -> {len(test_df):,}")

Trim bounds (train 1/99): $235,000 - $6,300,000
Train: 118,247 -> 115,941 | Test: 12,837 -> 12,570


In [12]:
EXCLUDE = [
    "ListingKey",              
    "CloseDate", "CloseMonth",  

    "ClosePrice",               # target var
    "ClosePrice_log",           # target in log form
    "ClosePrice_repaired",      # metadata

    "CountyOrParish",           # one-hots from 02_preprocessing
    "PostalCode",               

    "LivingArea",               # using LivingArea_log
    "LotSizeAcres",             # using LotSizeAcres_log

    "County_Los_Angeles",       # reference
    "SchoolDistrict",
    "BedBathRatio", 
    "PropertyAge", 
    "LotToLiving_log", 
    "LotToLiving",
    "City",
    "LivingAreaPerBedroom",
    "HasGarage",
    "AmenityCount"

]

feature = [c for c in df.columns if c not in EXCLUDE and not c.startswith("District_")]
print(f"Old Feature Set: {len(feature)} features"
    f"({sum(c.startswith('Zip_') for c in feature)} ZIP, "
    f"{sum(c.startswith('City_') for c in feature)} City)")


# Checks
assert not any("ClosePrice" in c for c in feature), "Target column leaked"
assert df[feature].isna().sum().sum() == 0, "Feature matrix contains NaNs"

X_train, X_test = train_df[feature], test_df[feature]
y_train, y_test = train_df[TARGET], test_df[TARGET]

Old Feature Set: 1185 features(705 ZIP, 407 City)


In [13]:
def fit_eval(model, use_log=True, X_tr=None, X_te=None):
    X_tr = X_train if X_tr is None else X_tr
    X_te = X_test if X_te is None else X_te

    ytr = np.log(y_train) if use_log else y_train
    model.fit(X_tr, ytr)

    pred_train = model.predict(X_tr)
    pred_test = model.predict(X_te)
    if use_log:
        pred_train, pred_test = np.exp(pred_train), np.exp(pred_test)

    return {
        "model": model,
        "pred_train": pred_train,
        "pred_test": pred_test,
        "train_r2": r2_score(y_train, pred_train),
        "test_r2": r2_score(y_test, pred_test),
    }

In [17]:
def log_result(model_name, version, train_r2, test_r2, change_note, target_space="raw(ClosePrice)", week=5, n_features=None):
    n_features = len(feature) if n_features is None else n_features
    os.makedirs(os.path.dirname(RESULTS), exist_ok=True)
    row = pd.DataFrame([{
        "week": week, "model": model_name, "version": version,
        "target": target_space, "train_months": TRAIN_MONTHS,
        "test_month": TEST_MONTH, "n_features": n_features,
        "train_r2": round(train_r2, 4), "test_r2": round(test_r2, 4),
        "change_note": change_note,
    }])
    if os.path.exists(RESULTS):
        log = pd.read_csv(RESULTS)
        same = ((log["model"] == model_name) & (log["version"] == version)
                & (log["week"] == week) & (log["test_month"] == TEST_MONTH)
                & (log["train_months"] == TRAIN_MONTHS))
        log = log[~same]
        log = pd.concat([log, row], ignore_index=True)
    else:
        log = row
    log.to_csv(RESULTS, index=False)
    print(f"logged: {model_name} {version} | week {week} | n_feat {n_features} | "
            f"train R2 {train_r2:.4f} | test R2 {test_r2:.4f}")

results_log = pd.read_csv(RESULTS)
base = results_log[(results_log["model"] == "LinearRegression") &
                    (results_log["version"] == "baseline")]
assert len(base) == 1, "Missing baseline row, run 03baseline_model.ipynb first."

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lin = fit_eval(LinearRegression(), use_log=False, X_tr=X_train_s, X_te=X_test_s)
print(f"linear (raw): train R^2 = {lin['train_r2']:.4f} | test R^2 = {lin['test_r2']:.4f}")

BASELINE_R2 = float(base['test_r2'].iloc[0])
assert abs(lin['test_r2'] - BASELINE_R2) < 0.01, "04 unable to reproduce 03's result"

linear (raw): train R^2 = 0.8158 | test R^2 = 0.8152


In [18]:
order = np.argsort(train_df[DATE].to_numpy())
Xtr = train_df.iloc[order][feature].reset_index(drop=True)
ytr = train_df.iloc[order][TARGET].reset_index(drop=True)   # raw ClosePrice target

per = train_df.iloc[order][DATE].dt.to_period("M").reset_index(drop=True)
monthly_splits = []
for vm in sorted(per.unique())[-4:]: #using recent 4 months
    tr_pos = np.flatnonzero((per < vm).to_numpy())
    va_pos = np.flatnonzero((per == vm).to_numpy())
    monthly_splits.append((tr_pos, va_pos))

## GB.A LightGBM

In [20]:
gb_a = fit_eval(
    LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=63,
                  random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    use_log=False, X_tr=train_df[feature], X_te=test_df[feature])

print(f"LightGBM A (raw): train {gb_a['train_r2']:.4f} | test {gb_a['test_r2']:.4f} "
      f"| gap {gb_a['train_r2'] - gb_a['test_r2']:.4f}")
print(f"(baseline = {BASELINE_R2:.4f} | RF.B to beat = 0.8699)")

log_result("LightGBM", "A", gb_a["train_r2"], gb_a["test_r2"],
           "defaults: 400 trees, lr=0.05, num_leaves=63, raw target", week=WEEK)

X_test_full = test_df_full[feature]
gb_pred_full = gb_a["model"].predict(X_test_full)
print("LightGBM A raw, dollar R^2:")
print(f"  contract (train-1/99 trimmed test): {gb_a['test_r2']:.4f}  [n={len(test_df):,}]")
print(f"  full untrimmed test (stricter):     {r2_score(test_df_full[TARGET], gb_pred_full):.4f}  [n={len(test_df_full):,}]")

LightGBM A (raw): train 0.9220 | test 0.8889 | gap 0.0331
(baseline = 0.8152 | RF.B to beat = 0.8699)
logged: LightGBM A | week 7 | n_feat 1185 | train R2 0.9220 | test R2 0.8889
LightGBM A raw, dollar R^2:
  contract (train-1/99 trimmed test): 0.8889  [n=12,570]
  full untrimmed test (stricter):     0.5811  [n=12,837]


### Interpret:

- between the trimmed and untrimmed set, there's a 0.3 spread
- currently the best performing model on the trimmed (1/99) train to test set
- however when we actually see the untrimmed (which includes the luxury), its 0.58 which characterizes the full market and is technically more realistic


GB.B

Tunes with recent 4 months

